# INHA AI Challenge Baseline

이 노트북은 공개 패키지 기준으로 데이터 확인, 환경 설치, 베이스라인 추론, 제출 CSV 생성을 실행하는 예시입니다.

참가자는 `../data/eval/images`의 시작 이미지와 `../data/eval/actions`의 action sequence를 조건으로 미래 영상을 생성하고, 생성 영상을 `../submission_kit`으로 feature CSV로 변환해 제출합니다.

In [1]:
import os
from pathlib import Path

cwd = Path.cwd()
if (cwd / 'challenge_kit').exists():
    BASELINE_ROOT = cwd
elif (cwd / 'baseline/challenge_kit').exists():
    BASELINE_ROOT = cwd / 'baseline'
    os.chdir(BASELINE_ROOT)
else:
    raise RuntimeError('baseline.ipynb는 open/baseline 또는 open 폴더에서 실행해 주세요.')
PACKAGE_ROOT = BASELINE_ROOT.parent
DATA_ROOT = PACKAGE_ROOT / 'data'
print('baseline root:', BASELINE_ROOT)
print('package root:', PACKAGE_ROOT)
print('eval images:', len(list((DATA_ROOT / 'eval/images').glob('*.png'))))
print('eval actions:', len(list((DATA_ROOT / 'eval/actions').glob('*.npy'))))
print('train exists:', (DATA_ROOT / 'train').exists())

baseline root: /home/dacon/Dacon/HDD_01/gimin/2026/인하대/baseline
package root: /home/dacon/Dacon/HDD_01/gimin/2026/인하대
eval images: 216
eval actions: 216
train exists: True


## 1. 학습 데이터 확인

학습 데이터는 `../data/train` 폴더에 포함되어 있습니다.

In [2]:
from pathlib import Path

train_root = Path('../data/train')
assert train_root.exists(), 'data/train 폴더가 없습니다.'
for path in list(train_root.glob('*/*'))[:10]:
    print(path)

../data/train/liuhuanjim013/so100_th
../data/train/paszea/so100_lego_2cam
../data/train/paszea/so100_whale_2
../data/train/paszea/so100_whale_3
../data/train/paszea/so100_lego
../data/train/paszea/so100_whale_4
../data/train/sihyun77/sihyun_3_17_2
../data/train/sihyun77/suho_3_17_1
../data/train/sihyun77/sihyun_3_17_5
../data/train/sihyun77/suho_3_17_3


## 2. 패키지 설치

베이스라인 코드는 `challenge_kit` 안에서 실행합니다.

In [3]:
import subprocess
import sys

print('Python:', sys.version)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    '-e', 'challenge_kit',
    '-e', 'challenge_kit/libs/dynamicrafter',
    '-e', 'shared_libs/video_utils',
])

Python: 3.8.18 (default, Sep 11 2023, 13:40:15) 
[GCC 11.2.0]


  DEPRECATION: Legacy editable install of video-utils==0.1.0 from file:///home/dacon/Dacon/HDD_01/gimin/2026/%EC%9D%B8%ED%95%98%EB%8C%80/baseline/shared_libs/video_utils (setup.py develop) is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  DEPRECATION: Legacy editable install of lvdm==0.0.0 from file:///home/dacon/Dacon/HDD_01/gimin/2026/%EC%9D%B8%ED%95%98%EB%8C%80/baseline/challenge_kit/libs/dynamicrafter (setup.py develop) is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected

0

## 3. 베이스라인 backbone 준비

제공된 `baseline_diffusion.ckpt`로 베이스라인 영상을 생성하려면 DynamiCrafter pretrained backbone이 필요합니다. 이미 생성한 mp4를 제출 CSV로 변환만 할 경우에는 이 단계가 필요하지 않습니다.

In [4]:
from pathlib import Path
from urllib.request import urlretrieve

backbone_path = Path('checkpoints/backbone.ckpt')
backbone_path.parent.mkdir(parents=True, exist_ok=True)
if not backbone_path.exists():
    url = 'https://huggingface.co/Doubiiu/DynamiCrafter_512/resolve/main/model.ckpt'
    print(f'Downloading {url} -> {backbone_path}')
    urlretrieve(url, backbone_path)
else:
    print(f'Already exists: {backbone_path}')

Already exists: checkpoints/backbone.ckpt


## 4. 선택: 베이스라인 학습

제공된 `baseline_diffusion.ckpt`를 그대로 사용할 경우 이 단계는 자동으로 건너뜁니다.

In [5]:
import os
import shutil
import subprocess
import sys

RUN_BASELINE_TRAINING = False # False (Default) : 학습 진행하지 않습니다.

if RUN_BASELINE_TRAINING:
    if shutil.which('bash') is None:
        raise RuntimeError('베이스라인 학습 스크립트 실행에는 bash가 필요합니다.')
    env = os.environ.copy()
    env['PYTHON_BIN'] = sys.executable
    env['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
    subprocess.check_call([
        'bash', 'scripts/train.sh',
        '--config', 'configs/train/inha_action_diffusion_11M.yaml',
        '--script', 'scripts/train_diffusion.py',
    ], cwd='challenge_kit', env=env)
else:
    print('Skip baseline training. Provided baseline_diffusion.ckpt will be used.')

Skip baseline training. Provided baseline_diffusion.ckpt will be used.


## 5. 베이스라인 영상 생성

아래 명령은 `backbone.ckpt`로 기본 모델을 구성하고, 제공된 `baseline_diffusion.ckpt`를 로드해 평가 입력 216개에 대한 mp4 영상을 생성합니다.

In [6]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, 'scripts/inference/generate_baseline_videos.py',
    '--checkpoint', '../checkpoints/baseline_diffusion.ckpt',
    '--challenge-root', '../../data/eval',
    '--prediction-root', '../../submission_kit/input_videos',
], cwd='challenge_kit')

AE working on z of shape (1, 4, 32, 32) = 4096 dimensions.


/home/dacon/anaconda3/envs/gimin_py38/lib/python3.8/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/dacon/anaconda3/envs/gimin_py38/lib/python3.8/site-packages/open_clip/factory.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the 

0

## 6. 제출 CSV 생성

아래 명령은 생성된 mp4 영상을 DINO/R3D/Action feature로 변환해 제출 CSV를 만듭니다. 참가자 자체 모델을 사용할 경우에도 `../submission_kit/input_videos/sample_000000.mp4` 형식으로 영상을 저장한 뒤 `../submission_kit`만 실행하면 됩니다.

In [7]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, 'make_submission_csv.py',
], cwd='../submission_kit')

/home/dacon/anaconda3/envs/gimin_py38/lib/python3.8/site-packages/lightning_fabric/utilities/cloud_io.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(

[feature csv] submission: wrote 4/216 samples
[feature csv] submission: wrote 8/216 samples
[feature csv] submission: wrote 12/216 samples
[feature csv] submission: wrote 16/216 samples
[feature csv] submission: wrote 20/216 samples
[feature csv] submission: wrote 24/216 samples
[feature csv] submission: wrote 28/216 samples
[feature csv] submission: wrote 32/216 samples
[feature csv] submission: wrote 36/216 samples
[feature csv] submission: wrote 40/216 samples
[feature csv] submission: wrote 44/216 samples
[feature csv] submission: wrote 48/216 samples
[feature csv] submission: wrote 52/216 samples
[feature csv] submission: wrote 56/216 samples
[feature csv] submission: wrote 60/216 samples
[feature csv] submission: wrote 64/216 samples
[feature csv] submission: wrote 68/216 samples
[feature csv] submission: wrote 72/216 samples
[feature csv] submission: wrote 76/216 samples
[feature csv] submission: wrote 80/216 samples
[feature csv] submission: wrote 84/216 samples
[feature csv] s

0

## 7. 제출 파일 확인

In [8]:
import pandas as pd

submission_path = PACKAGE_ROOT / 'submission_kit/submission_features.csv'
submission = pd.read_csv(submission_path)
print(submission.shape)
display(submission.head())
print(submission['feature_component'].value_counts())

(648, 3)


,sample_id,feature_component,feature_json
0,sample_000000,Video Feature Component,"[0.7257390022277832,0.011118000373244286,0.031..."
1,sample_000001,Video Feature Component,"[1.6645599603652954,0.24139399826526642,0.0587..."
2,sample_000002,Video Feature Component,"[1.1185450553894043,0.43237099051475525,0.0422..."
3,sample_000003,Video Feature Component,"[1.683419942855835,0.28567400574684143,0.05430..."
4,sample_000000,DINO Component,"[[-2.3287839889526367,-0.8862519860267639,-1.6..."


feature_component
Video Feature Component    216
DINO Component             216
Action Component           216
Name: count, dtype: int64
